In [1]:
!pip install sentence-transformers jsonlines > /dev/null

In [2]:
import torch
import numpy as np
import json
from torch.utils.data import Dataset, DataLoader
from sentence_transformers.cross_encoder import CrossEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
import os

MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
BATCH_SIZE = 16
NUM_EPOCHS = 3
BASE_PATH = '/kaggle/input/touche-task2-responses'

2025-05-21 12:31:36.921823: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747830697.148885      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747830697.214147      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
class AdDataset(Dataset):
    def __init__(self, responses_path, labels_path):
        self.data = []
        
        # Load responses
        with open(responses_path, 'r') as f:
            responses = [json.loads(line) for line in f]
        
        # Load labels
        with open(labels_path, 'r') as f:
            labels = {item['id']: item for item in [json.loads(line) for line in f]}
        
        # Merge data
        for resp in responses:
            label_info = labels.get(resp['id'], {'label': 0})
            self.data.append({
                'text': resp['response'],
                'label': label_info['label']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
train_dataset = AdDataset(
    os.path.join(BASE_PATH, 'responses-train.jsonl'),
    os.path.join(BASE_PATH, 'responses-train-labels.jsonl')
)

val_dataset = AdDataset(
    os.path.join(BASE_PATH, 'responses-validation.jsonl'),
    os.path.join(BASE_PATH, 'responses-validation-labels.jsonl')
)

In [5]:
model = CrossEncoder(MODEL_NAME, num_labels=1)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
from sentence_transformers import InputExample

train_samples = [
    InputExample(texts=[item['text']], label=item['label']) 
    for item in train_dataset
]

val_samples = [
    InputExample(texts=[item['text']], label=item['label']) 
    for item in val_dataset
]


In [7]:
def custom_collate_fn(batch):
    texts = [example.texts[0] for example in batch]
    labels = [float(example.label) for example in batch]
    
    # Tokenize single sentences (not pairs)
    tokenized = model.tokenizer(
        texts,
        padding=True,
        truncation='longest_first',
        return_tensors='pt',
        max_length=512
    )
    
    return tokenized, torch.tensor(labels)

train_dataloader = DataLoader(
    train_samples, 
    shuffle=True, 
    batch_size=BATCH_SIZE,
    collate_fn=custom_collate_fn
)


In [8]:
class BinaryEvaluator:
    def __init__(self, samples):
        self.samples = samples
        
    def __call__(self, model, output_path=None, epoch=-1, steps=-1, **kwargs):
        dataloader = DataLoader(
            self.samples,
            batch_size=BATCH_SIZE,
            collate_fn=custom_collate_fn
        )
        
        pred_labels = []
        true_labels = []
        
        model.model.eval()
        with torch.no_grad():
            for batch in dataloader:
                inputs, labels = batch
                outputs = model.model(**inputs.to(model.device))
                preds = torch.sigmoid(outputs.logits.squeeze()).cpu().numpy()
                pred_labels.extend([1 if p > 0.5 else 0 for p in preds])
                true_labels.extend(labels.numpy().astype(int))
        
        f1 = f1_score(true_labels, pred_labels)
        accuracy = accuracy_score(true_labels, pred_labels)
        
        # Print metrics
        print(f"\nValidation @ Epoch {epoch}:")
        print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
        
        # Return F1 score 
        return f1


In [9]:
model = CrossEncoder(
    'sentence-transformers/all-mpnet-base-v2',
    num_labels=1,
    max_length=512,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)


Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
evaluator = BinaryEvaluator(val_samples)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=NUM_EPOCHS,
    warmup_steps=100,
    output_path='./best_model',
    weight_decay=0.01,
    show_progress_bar=True
)

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/718 [00:00<?, ?it/s]


Validation @ Epoch 0:
Accuracy: 0.9899, F1: 0.9860


Iteration:   0%|          | 0/718 [00:00<?, ?it/s]


Validation @ Epoch 1:
Accuracy: 0.9923, F1: 0.9894


Iteration:   0%|          | 0/718 [00:00<?, ?it/s]


Validation @ Epoch 2:
Accuracy: 0.9929, F1: 0.9903


In [11]:
test_dataset = AdDataset(
    os.path.join(BASE_PATH, 'responses-test.jsonl'),
    os.path.join(BASE_PATH, 'responses-test-labels.jsonl')
)


In [12]:
test_texts = [item['text'] for item in test_dataset]
test_labels = [item['label'] for item in test_dataset]

test_loader = DataLoader(
    [InputExample(texts=[text], label=0) for text in test_texts],  # Dummy labels
    batch_size=BATCH_SIZE,
    collate_fn=custom_collate_fn
)

pred_probs = []
model.model.eval()
with torch.no_grad():
    for batch in test_loader:
        inputs, _ = batch  # Ignore dummy labels
        outputs = model.model(**inputs.to(model.device))
        preds = torch.sigmoid(outputs.logits.squeeze()).cpu().numpy()
        pred_probs.extend(preds)

pred_labels = [1 if p > 0.5 else 0 for p in pred_probs]

print(f"Test Accuracy: {accuracy_score(test_labels, pred_labels):.4f}")
print(f"Test F1 Score: {f1_score(test_labels, pred_labels):.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, pred_labels))

Test Accuracy: 0.9935
Test F1 Score: 0.9907

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1687
           1       0.99      0.99      0.99       913

    accuracy                           0.99      2600
   macro avg       0.99      0.99      0.99      2600
weighted avg       0.99      0.99      0.99      2600



In [13]:
!zip -r best_model.zip /kaggle/working/best_model

  adding: kaggle/working/best_model/ (stored 0%)
  adding: kaggle/working/best_model/tokenizer.json (deflated 71%)
  adding: kaggle/working/best_model/config.json (deflated 47%)
  adding: kaggle/working/best_model/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: kaggle/working/best_model/vocab.txt (deflated 53%)
  adding: kaggle/working/best_model/tokenizer_config.json (deflated 75%)
  adding: kaggle/working/best_model/special_tokens_map.json (deflated 85%)


In [14]:
!cp best_model.zip /kaggle/working/

cp: 'best_model.zip' and '/kaggle/working/best_model.zip' are the same file


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
